In [18]:
import os,sys,subprocess,datetime,pathlib,tempfile,shutil,zipfile,io,base64,hashlib,json,signal,time
os.environ['CUDA_VISIBLE_DEVICES']='0'
CUTOFF=datetime.datetime(2026,10,31,8,35,tzinfo=datetime.timezone.utc)
assert datetime.datetime.now(datetime.timezone.utc)<datetime.datetime(2026,10,31,8,20,tzinfo=datetime.timezone.utc)
assert 'CUBLAS_WORKSPACE_CONFIG' not in os.environ
hardware=subprocess.check_output(['nvidia-smi','--query-gpu=index,name,memory.used,temperature.gpu,clocks.sm','--format=csv,noheader,nounits'],text=True);print(hardware,flush=True)
assert all('T4' in r.split(',')[1] and float(r.split(',')[2])<1000 for r in hardware.strip().splitlines())
root=pathlib.Path(tempfile.mkdtemp(prefix='round2-final-',dir='/kaggle/working'));print('ROOT',root,flush=True)
raw=base64.b64decode('...');assert hashlib.sha256(raw).hexdigest()=='0d31c3c95c2fcd1bbc47706bfa137a3e18c24e0cca73f5a29bde0f4e1bf41857'
with zipfile.ZipFile(io.BytesIO(raw))as z:
 assert all(not pathlib.PurePosixPath(n).is_absolute()and '..'not in pathlib.PurePosixPath(n).parts for n in z.namelist());z.extractall(root)
manifest={'solution/swiglu_pack.py': '31548adc5021ca814156a162b3127adc6fa27d482c6aa13f0ba3c52687ab2e0e', 'solution/rmsnorm.py': '60bb02df363aab45cb9a32639850888d7203d784f83295bc713d4a93acd5b2c5', 'solution/test_bnb_provenance.py': 'f4d1ebdf69ff21026d6a2213117a42caa8500122ed53702918118102bdd4a409', 'solution/test_residual_norm.py': 'f42ed5da71aff5c6d3b740ef3a20407902c999933257b5b372eb38e7653f5793', 'solution/test_rotary.py': 'c180e3abb37df60c71755987db94f94e410adada269b6c68d94c7d1aedc85d9d', 'solution/attention_gqa.py': 'f4d2cdafe0ece75c4a2510ce44acfec14a6f819040e16e0dca3b282b55ab3dd1', 'solution/rotary.py': '488522f8332e5402a85b5df74a9caaa080004c8ca9ab588f82c0f0ca292b2677', 'solution/hybrid_mlp.py': '89421a52d5025bc1b35e53b0723ded7216551db4309ca6ef0440098e3ab3f199', 'solution/attention_adapter.py': '1add9019d245aba7872f79bee155a86d8bbaab8ba47f548c956f23d621827e14', 'solution/stage_options.py': '3aff930958f2f91c94ae26a3fa21e07e5714671a6a9d96450c0094c38dcf2afd', 'solution/head_loss.py': 'fd66b7eef815487e5a80d201eab8dfc5706d1a1d0e83813ea90dfb7121a087b7', 'solution/residual_norm.py': '6b453981e8570a9e448accde1422d4fbfc766d330d432ec6082a6f72d6de473c', 'solution/nf4.py': '94a4cd004cd83a09ec51794b9a05fd6d5da9d6e073f144b20e3e8a8cf9c93740', 'solution/test_rmsnorm.py': '639f8dca507c1ac54fd65f5ffd190d9af24653ee2b6c1732a92874e512739b74', 'solution/swiglu.py': 'e1dc639838c4c46ea0ed0aec3f0d904d360c92a91c601b943c9cfef85ca21c64', 'solution/__init__.py': '66db347091a0789aa93fa87f4b2ebd0c719cb98d2c701dd346d36ce7c55e3db0', 'solution/test_lora.py': '356a6f480c252f6f124a44f1da5ae3fd226a72388b92f7c61ffba757499b85fc', 'solution/packed.py': '263a8eb3bb7654aa2d152b78299308abbaed1b268c502b23ec56f88e48d595e2', 'solution/bnb_provenance.py': 'fe7a0c6fd7fa5597005e56f9be017020a5e25f4b3b816267c0abbc647f1c5905', 'solution/kcheck.py': 'f6b29c27738e727d2c10811dd47d51ac004b0a9f7f910f01c9dd16eb99970daa', 'solution/stage_adapter.py': 'f05b89a5c6b1d753bd5436c310c1ca32d27e13ce4a7a0e8478fa7131dc3c6194', 'solution/test_head_fallback.py': 'ef034f42c7e110e4dab6b3ac6f5c0db0ea16d5187ec9034698b93bfb5a703ec3', 'solution/lora.py': 'e923126c5fad64c23b08eefc42edc5cdf0092fb09230e14ee14d79a3c2e7a806', 'solution/test_nf4.py': '7363fd34c10a9b76703e2f47eaa84b5e02175c4d1b59085782de07e7f9377e66', 'solution/test_kcheck.py': '9d0e88d503f523366d7e5a0b02734a82efec9e043b7a5796ade03aaf7106b405', 'solution/native_half.py': 'd9c492c675968ae9e3160da5f4e47a123e21ba6d7b639cc2bb62da1d25318cec', 'solution/test_kernels.py': 'a45b842a4f22c1aefe404baeff1ac26511253452ecb0f633c985fbc9df8b252a', 'solution/test_head_loss.py': 'b8beecd53659f788312523f13930ed34ac5f7ce533095fb3049e7195233711f1', 'harness/_train.py': '45a62642dceab32cf3efcda504de87a9c25ea28b119b1806f56d14ff1146a89d', 'harness/check_rules.py': '879ab1861674f6e714eadd00aaf044235d98526dd75fab1035deec3401121ef5', 'harness/eval_adapter.py': '47cddbb8e4579c69f9535b099ff2371b1c3af08bfbd93d14bf79aadd74d38482', 'harness/score.py': '731c9b644dcc4069b0b069616f438064d1f7f376a202feb15201fd910ec688c5', 'harness/example_reference.py': '6a11055f4a1822e53971c228c6e3e3aa6b88edd9eb6db81a388e26a70a030ceb', 'harness/common.py': 'c83066f24f266dab62ebcc9c2d44f5a65902760b53da912e65ebfe511579c2db', 'harness/run_task.py': 'd935a8959f348977313f2eef36870ff2699b9ba013a7978918ff35e148968a79', 'task/task.yaml': 'dd70c739fcd143e06aef3bb1efcc9386f682bf1de26f30e08409d86a97ff47ce', 'task/prepare_data.py': 'a8543f11a55aff80dbdc714a0e62391244a5296616eea2e34dc2eb3ff3ec18f7', 'tests/test_loader.py': '3dad1bea2d33aab97e52b7cd44888d768bc9519af44fbe6cf3e96c260026632f', 'tests/test_trainer_init.py': 'b7608d62a40983bf7c9e5b270e0e527b641a837e462d6df2909fec88f607e821', 'tests/test_formats.py': '5d656ed50c59f794e934473d459ea85635f892ba028e192414474e9b5d65919e', 'tests/conftest.py': 'a82eabab468ea0ef1e003cccd7c3827ce2f4d00d09c32948a34fb48221f7337a', 'tests/test_callback.py': '1919303a3670c0a33abbfce9b33eab0913bb64a3a21c3312d86531c32b5bf490', 'tests/test_config.py': 'db808c44470d144b7cf18b7329492f41f76e37cfc24306e7cba0488bca9d787c', 'LICENSE': 'cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30', 'NOTICE': '2f0089a88e9623e5f6eba497796f279c2fdb3a15be1df0973e4d0055d35876e4', 'data/train.jsonl': '93d036d912bd81c337dc29893e13a76f36161d96f896ef4e02492e2f1a21a0dd', 'data/eval_public.jsonl': 'b7f85b78ba3186dce4deab7cfb8bf8dcf7998e1517c5749ac0f5005886c6f70e'}
assert all(hashlib.sha256((root/n).read_bytes()).hexdigest()==h for n,h in manifest.items())
(root/'source-manifest.json').write_text(json.dumps(manifest,indent=2))
def run(args,name,limit):
 remaining=(CUTOFF-datetime.datetime.now(datetime.timezone.utc)).total_seconds();assert remaining>0
 with(root/name).open('w')as log:
  p=subprocess.Popen([sys.executable,'-B',*args],cwd=root,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
  try:rc=p.wait(timeout=min(limit,remaining))
  except BaseException:
   os.killpg(p.pid,signal.SIGKILL);p.wait();raise
 print(name,'EXIT',rc,(root/name).read_text()[-7000:],flush=True);assert rc==0,name
try:
 run(['-m','pytest','solution','-q','-p','no:cacheprovider','--tb=short'],'solution-tests.log',300)
 run(['harness/run_task.py','--repeats','2','--timeout','240'],'harness.log',850)
 run(['harness/check_rules.py'],'checks.log',240)
 print('FINAL_CHECKS',(root/'runs/checks.json').read_text(),flush=True)
 print('FINAL_SUMMARY',json.dumps(json.loads((root/'runs/result.json').read_text())['summary']),flush=True)
 (root/'VALIDATED.txt').write_text('all stages exit0; inspect checks.json for each admission check')
except BaseException as e:
 (root/'error.txt').write_text(repr(e));print('FAILED',repr(e),flush=True);raise
finally:
 archive=pathlib.Path('/kaggle/working/decentrathon-round2-final.zip')
 with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED)as z:
  for p in root.rglob('*'):
   if p.is_file()and not any(n in p.parts for n in ('triton_cache','inductor_cache','__pycache__','cuda_cache')):z.write(p,p.relative_to(root))
 print('ARTIFACT',archive,archive.stat().st_size,'SHA256',hashlib.sha256(archive.read_bytes()).hexdigest(),flush=True)


0, Tesla T4, 3, 48, 300
1, Tesla T4, 0, 40, 300

ROOT /kaggle/working/round2-final-jsqvzk37
solution-tests.log EXIT 0 .......................................ss............sssssssssssssssssss [ 16%]
sssssssssss.sssssssssssssssssssssssssssss............................... [ 33%]
.....................s..s..s..s..s..s..s..s..s..s..s..s...........s..s.. [ 50%]
s..s..s..s..s..s..s..s..s..s..s..s..s..s..s..s..s..s..s..s..s..s..s..... [ 67%]
.......................................s..s..s..s..s..s..s...s..s..s..s. [ 84%]
.s..s..s..s..s..s..s..s..s..s..s..s..s..s..s........s..s............     [100%]
=============================== warnings summary ===============================
../../../usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:365: 14 warnings
  /usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:365: DeprecationWarning: `torch.jit.script_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
    warnings.warn(

solution/test_head_fallback.py::te